# Reply speed — is animated-elk the fastest to reply? (week 4, distributions)

Builds the Analysis 15 chart from `analysis-log.md`. Started as a curiosity: what kind of
distribution is reply time, and does it differ per author?

**Decisions, made with the student along the way (details in the log):**

- Reply = a message directly after someone else's message.
- Reply time is a *continuous* duration recorded in whole minutes. Both timestamps are
  rounded down, so a recorded gap of k minutes is really somewhere in (k−1, k+1): each gap is
  spread over that range (k = 0 keeps its order, so it lands in [0, 1)).
- Replies vs new conversations are two processes; the cut is the dip between them (≈ 392 min).
- Family: lognormal fits best (exponential clearly loses); per author, a mix of two lognormals
  (fast + slow) describes the shape better.
- Statistic for the claim: share of replies within 6 min (where fast and slow are equally likely).
- Tests fixed before looking: quiet-period test (60 min silence, ≥ 1.5× general share) and a
  shuffle test of the author labels within each day (threshold 10%).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from scipy import stats
from scipy.signal import argrelmin
from sklearn.mixture import GaussianMixture

from wa_analyzer.data import load_own_chat

ELK = "animated-elk"
FAST = 6  # minutes -- where the fast and slow process are equally likely
SEED = 42

own = load_own_chat()
# same fix as 06-10: `timestamp` is stored as strings; the `+00:00` is only a label
own["timestamp"] = pd.to_datetime(own["timestamp"], utc=True).dt.tz_localize(None)
own = own.sort_values("timestamp", kind="stable").reset_index(drop=True)
own["day"] = own["timestamp"].dt.date

print("seconds ever recorded:", (own["timestamp"].dt.second != 0).any())

## Reply gaps, with the minute rounding spread out

`recorded` is the whole-minute gap from the export. The true gap is `recorded + (s2 − s1)`,
with `s1`, `s2` the unknown seconds of both timestamps (uniform). Spreading only within one
minute (or adding +0.5) piles up identical values and creates a fake spike at 1–2 minutes.

In [ ]:
rng = np.random.default_rng(SEED)
recorded = own["timestamp"].diff().dt.total_seconds() / 60
s1, s2 = rng.uniform(0, 1, len(own)), rng.uniform(0, 1, len(own))
own["gap"] = np.where(recorded == 0, np.abs(s2 - s1), recorded + s2 - s1)

previous_author = own["author"].shift()
author_changed = own["author"].ne(previous_author) & recorded.notna()
print(
    f"{author_changed.sum():,} reply gaps, {(recorded[author_changed] == 0).mean():.0%} recorded as 0 minutes"
)

## Where replies end and new conversations start

The log-gap density has two bumps (replies / new conversations). The cut is the local minimum
between them, searched between 30 min and 50 h.

In [ ]:
log_gaps = np.log10(own.loc[author_changed, "gap"].clip(lower=0.01))
grid = np.linspace(log_gaps.min(), log_gaps.max(), 600)
density = stats.gaussian_kde(log_gaps)(grid)
dips = [10 ** grid[i] for i in argrelmin(density)[0] if 30 < 10 ** grid[i] < 3000]
CUT = round(dips[0])
print(f"dip between the bumps: {CUT} min ({CUT / 60:.1f} h)")

is_reply = author_changed & (own["gap"] > 0) & (own["gap"] < CUT)
replies = own[is_reply]
own["conversation"] = (recorded.fillna(np.inf) >= CUT).cumsum()
replies = own[is_reply]
print(f"{len(replies):,} replies within a conversation")

## Which family? (start fixed at 0 — a waiting time cannot start later)

Predicted before fitting: exponential would mean a steady random reply rate (median ≈ 0.69 × mean);
the data's median is far below that, so a heavier-tailed family should win.

In [ ]:
candidates = {
    "lognormal": stats.lognorm,
    "weibull": stats.weibull_min,
    "gamma": stats.gamma,
    "exponential": stats.expon,
}
rows = []
for name, family in candidates.items():
    fitted = family(*family.fit(replies["gap"], floc=0))
    rows.append(
        {
            "family": name,
            "log_likelihood": fitted.logpdf(replies["gap"]).sum(),
            "ks": stats.kstest(replies["gap"], fitted.cdf).statistic,
        }
    )
pd.DataFrame(rows).sort_values("log_likelihood", ascending=False).round(3)

## Per author: typical reply time, with uncertainty from resampling whole days

Reply gaps are not independent (bursts, time of day), so the bootstrap resamples **days**, not gaps.

In [ ]:
def speed(gaps):
    return np.exp(np.log(gaps).mean()), np.median(gaps), (gaps < FAST).mean()


days = np.array(sorted(replies["day"].unique()))
by_day = dict(tuple(replies.groupby("day")))
boot = []
for _ in range(1000):
    sample = pd.concat([by_day[d] for d in rng.choice(days, len(days))])
    boot.append({a: speed(g["gap"].to_numpy()) for a, g in sample.groupby("author")})

rows = []
for author, frame in replies.groupby("author"):
    geo_mean, median, within = speed(frame["gap"].to_numpy())
    draws = np.array([b[author] for b in boot])
    rows.append(
        {
            "author": author,
            "replies": len(frame),
            "typical_min": geo_mean,
            "typical_lo": np.percentile(draws[:, 0], 2.5),
            "typical_hi": np.percentile(draws[:, 0], 97.5),
            "median_min": median,
            f"within_{FAST}": within,
            "within_lo": np.percentile(draws[:, 2], 2.5),
            "within_hi": np.percentile(draws[:, 2], 97.5),
        }
    )
per_author = pd.DataFrame(rows).sort_values("typical_min")

fastest_typical = np.mean(
    [all(b[ELK][0] < b[a][0] for a in b if a != ELK) for b in boot]
)
fastest_within = np.mean(
    [all(b[ELK][2] > b[a][2] for a in b if a != ELK) for b in boot]
)
print(
    f"{ELK} fastest of all 9 in {fastest_typical:.0%} (typical time) / {fastest_within:.0%} (within {FAST} min) of 1,000 resamples"
)
per_author.round(3)

### Does "fastest" depend on the 6-minute threshold?

In [ ]:
sweep = []
for threshold in [1, 2, 3, 6, 10, 30, 60]:
    shares = (
        replies.groupby("author")["gap"]
        .apply(lambda g: (g < threshold).mean())
        .sort_values(ascending=False)
    )
    others = shares.drop(ELK)
    sweep.append(
        {
            "within_min": threshold,
            ELK: shares[ELK],
            "best_other": others.max(),
            "best_other_author": others.idxmax(),
            "elk_rank": list(shares.index).index(ELK) + 1,
        }
    )
pd.DataFrame(sweep).round(2)

## Quiet-period test (fixed before looking)

After ≥ 60 min of silence nobody is likely to have the app open. If the notification bell is
what makes animated-elk fast, they should (1) be the first to reply ≥ 1.5× as often as their
general share of replies, and (2) keep their speed advantage.

In [ ]:
QUIET = 60
opener = own["gap"].where(recorded.notna(), np.inf) >= QUIET
burst = opener.cumsum()
first_replies = []
for _, frame in own[burst > 0].groupby(burst[burst > 0]):
    opener_author = frame["author"].iloc[0]
    if opener_author == ELK:
        continue
    others = frame[frame["author"] != opener_author]
    if not others.empty and is_reply[others.index[0]]:
        first_replies.append(others.index[0])
first = own.loc[first_replies]
general = own[is_reply & (previous_author != ELK) & ~opener]

share_first, share_general = (
    (first["author"] == ELK).mean(),
    (general["author"] == ELK).mean(),
)
first_days = first["day"].unique()
first_by_day = dict(tuple(first.groupby("day")))
ratios = [
    (
        pd.concat([first_by_day[d] for d in rng.choice(first_days, len(first_days))])[
            "author"
        ]
        == ELK
    ).mean()
    / share_general
    for _ in range(1000)
]


def advantage(frame):
    return (
        frame.loc[frame["author"] != ELK, "gap"].median()
        / frame.loc[frame["author"] == ELK, "gap"].median()
    )


print(
    f"first to reply: {share_first:.3f} vs general {share_general:.3f} -> ratio {share_first / share_general:.2f} "
    f"(95% CI {np.percentile(ratios, 2.5):.2f}-{np.percentile(ratios, 97.5):.2f}; criterion 1.5)"
)
print(
    f"speed advantage after silence: {advantage(first):.1f}x, within conversations: {advantage(general):.1f}x"
)

## Shuffle test (teacher's suggestion, fixed before looking)

Shuffle *who* sent each reply, only among replies of the same day (so busy, fast days stay busy and
fast), 2,000 times. Holds up if fewer than 10% of shuffles reach animated-elk's real share within 6 min.
Shuffling within conversations is the sensitivity check; shuffling everything is shown for reference.

In [ ]:
is_fast = (replies["gap"] < FAST).to_numpy()
is_elk = (replies["author"] == ELK).to_numpy()
observed = is_fast[is_elk].mean()


def shuffled_shares(group_column, n=2000):
    shuffle_rng = np.random.default_rng(SEED)
    groups = [
        np.flatnonzero(replies[group_column].to_numpy() == value)
        for value in replies[group_column].unique()
    ]
    groups = [g for g in groups if len(g) > 1]
    shares = np.empty(n)
    for i in range(n):
        labels = is_elk.copy()
        for g in groups:
            labels[g] = shuffle_rng.permutation(labels[g])
        shares[i] = is_fast[labels].mean()
    return shares


null_day = shuffled_shares("day")
null_conversation = shuffled_shares("conversation")
null_everything = np.array(
    [is_fast[np.random.default_rng(i).permutation(is_elk)].mean() for i in range(2000)]
)
for name, null in [
    ("within day", null_day),
    ("within conversation", null_conversation),
    ("everything", null_everything),
]:
    p = (np.sum(null >= observed) + 1) / (len(null) + 1)
    print(f"{name:20s} mean {null.mean():.3f}, highest {null.max():.3f}, p = {p:.4f}")
print(
    f"observed {ELK}: {observed:.3f}, rest of the group: {is_fast[~is_elk].mean():.3f}"
)

## The chart

Two-lognormal mix per group (thick), the data as thin lines, the 6-minute line with the tested
numbers. The bell is drawn from the macOS emoji font (falls back to text elsewhere).

In [ ]:
ORANGE, GREY = "#d95f02", "#555555"
REST = "rest of the group"
groups = {
    ELK: replies.loc[is_elk, "gap"].to_numpy(),
    REST: replies.loc[~is_elk, "gap"].to_numpy(),
}
fits = {}
for name, gaps in groups.items():
    mixture = GaussianMixture(2, random_state=0, n_init=3).fit(np.log(gaps)[:, None])
    order = np.argsort(mixture.means_.ravel())
    fits[name] = (
        mixture.weights_[order],
        mixture.means_.ravel()[order],
        np.sqrt(mixture.covariances_.ravel()[order]),
    )
share_fast = {name: (gaps < FAST).mean() for name, gaps in groups.items()}


def bell_image():
    try:
        from PIL import Image, ImageDraw, ImageFont

        font = ImageFont.truetype("/System/Library/Fonts/Apple Color Emoji.ttc", 160)
        image = Image.new("RGBA", (180, 180), (0, 0, 0, 0))
        ImageDraw.Draw(image).text((10, 10), "\N{BELL}", font=font, embedded_color=True)
        return np.asarray(image.crop(image.getbbox()))
    except OSError:
        return None


grid = np.logspace(-2, np.log10(CUT), 400)
bins = np.logspace(-2, np.log10(CUT), 45)
fig = plt.figure(figsize=(11, 6.6))
ax = fig.add_axes([0.08, 0.35, 0.88, 0.52])
top = 0
for name, colour in [(REST, GREY), (ELK, ORANGE)]:
    heights, _ = np.histogram(np.log10(groups[name]), bins=np.log10(bins), density=True)
    top = max(top, heights.max())
    ax.stairs(heights, bins, color=colour, lw=0.7, alpha=0.18)
    weights, means, sds = fits[name]
    curve = sum(
        weights[i] * stats.norm(means[i], sds[i]).pdf(np.log(grid)) * np.log(10)
        for i in range(2)
    )
    ax.plot(grid, curve, color=colour, lw=2.4)
ax.text(
    0.07,
    0.40,
    f"{ELK}\n{len(groups[ELK]):,} replies",
    color=ORANGE,
    fontsize=10,
    weight="bold",
    ha="right",
)
ax.text(
    40,
    0.36,
    f"{REST} (8 people)\n{len(groups[REST]):,} replies",
    color=GREY,
    fontsize=10,
    weight="bold",
)
ax.axvline(FAST, color="k", lw=0.9, ls="--")
ax.text(
    FAST * 0.88,
    top * 1.07,
    f"replies within {FAST} min",
    fontsize=9.5,
    va="top",
    ha="right",
    weight="bold",
)
ax.text(
    FAST * 0.88,
    top * 1.0,
    f"{ELK}  {share_fast[ELK]:.0%}",
    fontsize=9.5,
    va="top",
    ha="right",
    color=ORANGE,
    weight="bold",
)
ax.text(
    FAST * 0.88,
    top * 0.93,
    f"{REST}  {share_fast[REST]:.0%}",
    fontsize=9.5,
    va="top",
    ha="right",
    color=GREY,
)
others_within = per_author.loc[per_author["author"] != ELK, f"within_{FAST}"]
ax.text(
    FAST * 0.88,
    top * 0.86,
    f"each of the other 8: {others_within.min():.0%}–{others_within.max():.0%}",
    fontsize=9,
    va="top",
    ha="right",
    color=GREY,
    style="italic",
)
ax.set_xscale("log")
ax.set_xlim(0.01, CUT)
ax.set_ylim(0, top * 1.1)
ax.set_xticks(
    [0.01, 0.1, 1, 6, 60, 360], ["0.6 s", "6 s", "1 min", "6 min", "1 h", "6 h"]
)
ax.minorticks_off()
ax.set_xlabel("reply time (log scale)")
ax.set_ylabel("probability density (per log₁₀-minute)")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

title_y, left = 0.935, 0.08
before = fig.text(
    left,
    title_y,
    "The group's fastest thumbs belong to the only one with the",
    fontsize=15,
    weight="bold",
    va="center",
)
before_end = (
    before.get_window_extent(fig.canvas.get_renderer())
    .transformed(fig.transFigure.inverted())
    .x1
)
bell = bell_image()
if bell is None:
    fig.text(
        before_end + 0.006, title_y, "bell on", fontsize=15, weight="bold", va="center"
    )
else:
    fig.add_artist(
        AnnotationBbox(
            OffsetImage(bell, zoom=0.15),
            (before_end + 0.018, title_y),
            xycoords="figure fraction",
            frameon=False,
        )
    )
    fig.text(before_end + 0.037, title_y, "on", fontsize=15, weight="bold", va="center")

ranks = pd.DataFrame(sweep)["elk_rank"]
notes = [
    f"Fastest of all 9 in {fastest_within:.0%} of 1,000 resamples of days"
    + (", and at every threshold from 1 to 60 min." if (ranks == 1).all() else "."),
    f"Shuffle test: {ELK}'s name was shuffled among all replies of the same day, 2,000 times (seed {SEED}).",
    f"A random '{ELK}' replies within {FAST} min {null_day.mean():.0%} of the time on average (highest: {null_day.max():.0%}) — "
    f"the real {observed:.0%} was never reached (p < 0.001; threshold set beforehand: 10%).",
    f"Shuffling within conversations gives the same result. Part of the gap is context: {ELK} is active on faster days "
    f"({null_everything.mean():.0%} → {null_day.mean():.0%}).",
    f"Reply = message directly after someone else's, within {CUT} min (the dip between 'reply' and 'new conversation').",
    "Timestamps are whole minutes; each gap is spread over its possible ±1 min.",
    f"Notifications always on: known from the group, not measured. The data shows that {ELK} is faster, not why.",
    "Thick lines: fitted mix of two lognormals (a fast and a slow process); thin lines: the data. Group chat Aug 2020 – Sep 2026.",
]
fig.text(
    left,
    0.03,
    "\n".join(f"•  {n}" for n in notes),
    fontsize=8,
    color="0.3",
    va="bottom",
    linespacing=1.6,
)
fig.savefig("reply-speed-elk-vs-rest-draft.png", dpi=120)
plt.show()

## Supporting chart (not for the reader): share of replies per process

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 2.8))
for row, (name, colour) in enumerate([(ELK, ORANGE), (REST, GREY)]):
    weights, means, _ = fits[name]
    ax.barh(row, weights[0], color=colour, height=0.6)
    ax.barh(row, weights[1], left=weights[0], color=colour, alpha=0.3, height=0.6)
    ax.text(
        weights[0] / 2,
        row,
        f"{weights[0]:.0%} fast\n(median {np.exp(means[0]):.1f} min)",
        ha="center",
        va="center",
        color="white",
        fontsize=9,
        weight="bold",
    )
    ax.text(
        weights[0] + weights[1] / 2,
        row,
        f"{weights[1]:.0%} slow\n({np.exp(means[1]):.0f} min)",
        ha="center",
        va="center",
        fontsize=8,
    )
ax.set_yticks([0, 1], [ELK, REST])
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xticks([])
ax.set_title(
    "Model: share of replies in fast vs slow mode (two-lognormal mix)", fontsize=10
)
for spine in ["top", "right", "bottom"]:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
fig.savefig("reply-speed-model-share.png", dpi=120)
plt.show()